In [3]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers

In [4]:
# Paramètres
n_timesteps = 50
beta_t = np.linspace(0.0001, 0.02, n_timesteps).astype(np.float32)
alpha_t = 1- beta_t
alpha_bar_t = np.cumprod(alpha_t)
beta_bar_t = 1 - alpha_bar_t

# Génération des données d'entraînement
print("Génération des données...")
x = np.random.uniform(-2, 2, 50000)
y = x ** 2
real_data = np.stack([x, y], axis=1)

# Préparation du dataset d'entraînement
X_noisy, X_timesteps, y_noise = [], [], []

for i in range(len(real_data)):
    t = np.random.randint(0, n_timesteps)
    noise = np.random.randn(1,2)
    noisy = np.sqrt(alpha_bar_t[t])*real_data[i:i+1]+np.sqrt(1-alpha_bar_t[t])*noise
    
    X_noisy.append(noisy)
    X_timesteps.append([t / n_timesteps])  # Normalisation du timestep
    y_noise.append(noise)

X_noisy = np.array(X_noisy).reshape((50000,2))
X_timesteps = np.array(X_timesteps)
y_noise = np.array(y_noise).reshape((50000,2))

Génération des données...


In [5]:
# Création et entraînement du modèle
print("Création du modèle...")

data_input = layers.Input(shape=(2,), name='data')
t_input = layers.Input(shape=(1,), name='timestep')

# Embedding du timestep
t_emb = layers.Dense(32, activation='relu')(t_input)

# Fusion des inputs
x = layers.Concatenate()([data_input, t_emb])

# Réseau simple
x = layers.Dense(128, activation='relu')(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dense(64, activation='relu')(x)

# Prédiction du bruit
noise_pred = layers.Dense(2)(x)
    
model = keras.Model(inputs=[data_input, t_input], outputs=noise_pred)

model.compile(optimizer=keras.optimizers.Adam(0.001), loss='mse')

Création du modèle...


In [6]:
print("Entraînement du modèle...")
model.fit([X_noisy, X_timesteps],y_noise,epochs=30,batch_size=512,validation_split=0.1,verbose=1)

Entraînement du modèle...
Epoch 1/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.7273 - val_loss: 0.6430
Epoch 2/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.6067 - val_loss: 0.5932
Epoch 3/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5656 - val_loss: 0.5629
Epoch 4/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5435 - val_loss: 0.5527
Epoch 5/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5347 - val_loss: 0.5413
Epoch 6/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5279 - val_loss: 0.5360
Epoch 7/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5244 - val_loss: 0.5340
Epoch 8/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5233 - val_loss: 0.5316
Epoch 9/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5197 - val_loss: 0.5281
Epoch 10/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5163 - val_loss: 0.5271
Epoch 11/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5154 - val_loss: 0.5311
Epoch 12/30
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/st

In [ ]:
print("\nGénération de nouveaux échantillons...")
n_samples = 500

# Étape 1 : on part d'un bruit gaussien


# Étape 2 : débruitage progressif


# Étape 3 : on représente les points générés par le modèle de diffusion
#           et on compare avec la courbe d'équation y = x^2
